# Project 27: Insurance Claims & Settlement Analysis
**Industry:** Insurance / Financial Services  
**Business scenario:** Study insurance claims, claim amounts, claim types, and
settlement status to understand claim patterns and processing performance.  
**Dataset file:** 27_insurance_claims.csv

In [ ]:
# Importing required libraries
import pandas as pd
import numpy as np


Question 1. Load the claims dataset and inspect columns, dimensions, and data types.

In [145]:


df = pd.read_csv("27_insurance_claims.csv") # Reads the file into DataFrame (stores data in tabular format)
print(df) # Displays the dataset (for inspection)
print("Dataset Dimensions/Shape (rows, columns):", df.shape) #Tells dimensions (number of rows and columns)

print("\n--Column Names:")
print(list(df.columns)) # lists Column Names

print("\n--Data Types:")
print(df.dtypes) # shows each column's data type

print("\n--Dataset Information:")
df.info() # shows non-null counts per column, which hints at missing values


     Claim_ID  Claim_Date Insurance_Type Claim_Type  Claim_Amount  \
0    CL000000  2026-09-17          Motor    Medical     393052.81   
1    CL000001  2026-03-19       Property      Other     268590.57   
2    CL000002  2025-08-09         Health      Theft     320110.30   
3    CL000003  2025-08-09       Property    Medical     488247.47   
4    CL000004  2026-08-11          Motor       Loss     483651.10   
..        ...         ...            ...        ...           ...   
756  CL000336  2026-01-26           Life       Loss      43505.89   
757  CL000155  2026-09-01           Life      Theft     198735.51   
758  CL000528  2026-03-14       Property       Loss     270681.34   
759  CL000736  2025-06-30           Life     Damage     445843.86   
760  CL000097  2025-10-29           Life    Medical       2513.94   

     Settlement_Days Claim_Status Customer_ID  
0              117.0     Approved    CUS00060  
1                2.0     Rejected    CUS00254  
2               63.0     Re

Question 2. Identify missing values and duplicate Claim_ID records.

In [146]:

# Check missing values
print("Missing values per column:\n", df.isnull().sum())

#  Check total missing values
print("\nTotal Missing Values:", df.isnull().sum().sum())

#  Check duplicate Claim_ID records
print("\nDuplicate Claims:", df["Claim_ID"].duplicated().sum())

#  Checking the duplicate values. Duplicate claims  should be removed only after checking that they are genuinely repeated records.
duplicates = df[df["Claim_ID"].duplicated(keep=False)]
print("\n",duplicates)

# Remove duplicates (keep the first occurrence of each Claim_ID)
df = df.drop_duplicates(subset="Claim_ID", keep="first").reset_index(drop=True)
print("\nShape after removing duplicates:", df.shape)

Missing values per column:
 Claim_ID            0
Claim_Date          0
Insurance_Type      0
Claim_Type          0
Claim_Amount       18
Settlement_Days    18
Claim_Status        0
Customer_ID         0
dtype: int64

Total Missing Values: 36

Duplicate Claims: 11

      Claim_ID  Claim_Date Insurance_Type Claim_Type  Claim_Amount  \
97   CL000097  2025-10-29           Life    Medical       2513.94   
133  CL000133  2025-11-25       Property     Damage     478473.08   
155  CL000155  2026-09-01           Life      Theft     198735.51   
250  CL000250  2025-11-23         Travel     Damage     473970.39   
299  CL000299  2025-07-06       Property   Accident      74221.87   
336  CL000336  2026-01-26           Life       Loss      43505.89   
357  CL000357  2025-03-25       Property     Damage      30370.63   
506  CL000506  2026-01-17         Travel     Damage     188034.07   
528  CL000528  2026-03-14       Property       Loss     270681.34   
680  CL000680  2025-04-08       Property   

Question 3. Handle missing Claim_Amount or Settlement_Days values using a justified strategy.

In [147]:
# Strategy chosen: Fill the missing values with the 'median' of the same group as it is not distorted by the skewed data.
# Median amount within the same Insurance_Type
df["Claim_Amount"] = df["Claim_Amount"].fillna(
    df.groupby("Insurance_Type")["Claim_Amount"].transform("median")
)
# Median settlement days within the same Claim_Type
df["Settlement_Days"] = df["Settlement_Days"].fillna(
    df.groupby("Claim_Type")["Settlement_Days"].transform("median")
)

# If a whole group was empty, use the overall median
df["Claim_Amount"] = df["Claim_Amount"].fillna(df["Claim_Amount"].median())
df["Settlement_Days"] = df["Settlement_Days"].fillna(df["Settlement_Days"].median())

print("Missing after cleaning:\n", df[["Claim_Amount", "Settlement_Days"]].isna().sum())

Missing after cleaning:
 Claim_Amount       0
Settlement_Days    0
dtype: int64


Question 4. Convert Claim_Date to datetime and extract month and year

In [148]:

df["Claim_Date"] = pd.to_datetime(
    df["Claim_Date"],
    errors="coerce" # converts invalid dates into NaT(Not a Time)
 )
#Now extract month and year
df["Month"] = df["Claim_Date"].dt.month
df["Year"] = df["Claim_Date"].dt.year

print(df[["Claim_Date", "Month", "Year"]])
print(df.dtypes["Claim_Date"])

#If invalid dates exist, investigate them rather than silently treating them as valid dates.
print("\nInvalid Claim Dates:")
print(df["Claim_Date"].isnull().sum())

    Claim_Date  Month  Year
0   2026-09-17      9  2026
1   2026-03-19      3  2026
2   2025-08-09      8  2025
3   2025-08-09      8  2025
4   2026-08-11      8  2026
..         ...    ...   ...
745 2025-05-02      5  2025
746 2026-03-06      3  2026
747 2026-07-20      7  2026
748 2026-01-20      1  2026
749 2026-09-25      9  2026

[750 rows x 3 columns]
datetime64[ns]

Invalid Claim Dates:
0


Question 5. Calculate total claims, approved claims, rejected claims, and approval percentage.

In [149]:

total_claims = len(df) # total number of rows

# Counting approved and rejected claims
approved_claims = (
    df["Claim_Status"].str.strip().str.lower() == "approved"
).sum()

rejected_claims = (
    df["Claim_Status"].str.strip().str.lower() == "rejected"
).sum()

# Calculating approval percentage
approval_percent=(approved_claims / total_claims) * 100 if total_claims > 0 else 0

print("\nCLAIM STATUS SUMMARY")
print("Total Claims:", total_claims)
print("Approved Claims:", approved_claims)
print("Rejected Claims:", rejected_claims)
print(f"Approval Percentage: {approval_percent:.2f}%")





CLAIM STATUS SUMMARY
Total Claims: 750
Approved Claims: 506
Rejected Claims: 123
Approval Percentage: 67.47%


Question 6. Calculate the total and average claim amount for approved claims

In [150]:
approved_df = df[
    df["Claim_Status"].str.strip().str.lower() == "approved"
]
total_approved_amount = approved_df["Claim_Amount"].sum()
average_approved_amount = approved_df["Claim_Amount"].mean()
print("\nAPPROVED CLAIM ANALYSIS:")
print(f"Total Approved Amount: {total_approved_amount:,.2f}")
print(f"Average Approved Amount: {average_approved_amount:,.2f}")




APPROVED CLAIM ANALYSIS:
Total Approved Amount: 124,321,776.76
Average Approved Amount: 245,695.21


Question 7. Rank insurance types by number of claims and total claim amount

In [151]:

insurance_claim_count = df.groupby(
    "Insurance_Type"
)["Claim_ID"].count().sort_values(
    ascending=False
)

print("\nCLAIMS BY INSURANCE TYPE")
print(insurance_claim_count)

insurance_claim_amount = df.groupby(
    "Insurance_Type"
)["Claim_Amount"].sum().sort_values(
    ascending=False
)

print("\nTOTAL CLAIM AMOUNT BY INSURANCE TYPE")
print(insurance_claim_amount)



CLAIMS BY INSURANCE TYPE
Insurance_Type
Travel      164
Health      152
Property    150
Life        148
Motor       136
Name: Claim_ID, dtype: int64

TOTAL CLAIM AMOUNT BY INSURANCE TYPE
Insurance_Type
Travel      42283704.04
Health      37301309.86
Motor       36065324.98
Property    35977607.97
Life        34804809.93
Name: Claim_Amount, dtype: float64


Question 8. Compare average settlement time across claim types.

In [152]:

settlement_comparison = df.groupby(
    "Claim_Type"
)["Settlement_Days"].mean().sort_values(
    ascending=False
)

print("\nAVERAGE SETTLEMENT DAYS BY CLAIM TYPE")
print(settlement_comparison)



AVERAGE SETTLEMENT DAYS BY CLAIM TYPE
Claim_Type
Loss        66.023256
Accident    63.712000
Theft       63.585938
Medical     61.131783
Other       57.765217
Damage      51.983871
Name: Settlement_Days, dtype: float64


Question 9. Analyze approval and rejection percentages by insurance type.



In [153]:
df["Claim_Status_Clean"] = (
    df["Claim_Status"].str.strip().str.title()
)

status_comparison = pd.crosstab(
    df["Insurance_Type"],
    df["Claim_Status_Clean"],
    normalize="index"
) * 100

print("\nAPPROVAL AND REJECTION PERCENTAGES")
print(status_comparison.round(2))




APPROVAL AND REJECTION PERCENTAGES
Claim_Status_Clean  Approved  Pending  Rejected
Insurance_Type                                 
Health                 65.13    16.45     18.42
Life                   68.92    14.19     16.89
Motor                  70.59    16.18     13.24
Property               68.00    16.67     15.33
Travel                 65.24    17.07     17.68


Question 10. Identify the 10 highest-value claims.

In [154]:

top_10_claims = df.nlargest(
    10,
    "Claim_Amount"
)

print("\nTOP 10 HIGHEST-VALUE CLAIMS")

print(
    top_10_claims[
        [
            "Claim_ID",
            "Insurance_Type",
            "Claim_Type",
            "Claim_Amount",
            "Claim_Status"
        ]
    ]
)
df.nlargest(10, "Claim_Amount")



TOP 10 HIGHEST-VALUE CLAIMS
     Claim_ID Insurance_Type Claim_Type  Claim_Amount Claim_Status
209  CL000209           Life    Medical     499442.63     Approved
475  CL000475           Life   Accident     499200.44     Approved
141  CL000141           Life   Accident     498956.30     Approved
77   CL000077         Travel     Damage     498828.66     Approved
546  CL000546         Travel      Theft     498721.83     Approved
589  CL000589         Health    Medical     498611.73     Approved
320  CL000320           Life       Loss     498416.53     Rejected
117  CL000117         Health    Medical     498016.11     Approved
5    CL000005       Property      Theft     497407.87      Pending
511  CL000511          Motor     Damage     496788.69     Approved


,Claim_ID,Claim_Date,Insurance_Type,Claim_Type,Claim_Amount,Settlement_Days,Claim_Status,Customer_ID,Month,Year,Claim_Status_Clean
209,CL000209,2025-06-08,Life,Medical,499442.63,110.0,Approved,CUS00181,6,2025,Approved
475,CL000475,2025-05-07,Life,Accident,499200.44,112.0,Approved,CUS00049,5,2025,Approved
141,CL000141,2026-05-03,Life,Accident,498956.30,107.0,Approved,CUS00062,5,2026,Approved
77,CL000077,2025-07-02,Travel,Damage,498828.66,40.0,Approved,CUS00398,7,2025,Approved
546,CL000546,2026-06-07,Travel,Theft,498721.83,11.0,Approved,CUS00085,6,2026,Approved
589,CL000589,2026-10-18,Health,Medical,498611.73,11.0,Approved,CUS00133,10,2026,Approved
320,CL000320,2025-10-31,Life,Loss,498416.53,55.0,Rejected,CUS00125,10,2025,Rejected
117,CL000117,2026-09-14,Health,Medical,498016.11,16.0,Approved,CUS00401,9,2026,Approved
5,CL000005,2025-11-29,Property,Theft,497407.87,37.0,Pending,CUS00243,11,2025,Pending
511,CL000511,2025-01-15,Motor,Damage,496788.69,92.0,Approved,CUS00052,1,2025,Approved


Question 11. Use a set to identify unique claim categories and a dictionary to store claim counts by status

In [155]:

categories = {
    "Medical",
    "Vehicle",
    "Medical",
    "Life"
}
print(categories)

unique_categories = set(
 df["Claim_Type"].dropna()
)
print("\nUNIQUE CLAIM CATEGORIES")
print(unique_categories)

# Ensure the column exists even if prior cells failed to run
if "Claim_Status_Clean" not in df.columns:
    df["Claim_Status_Clean"] = df["Claim_Status"].str.strip().str.title()

claim_status_dictionary = df[
    "Claim_Status_Clean"
].value_counts().to_dict()

print("\nCLAIM COUNTS BY STATUS")
print(claim_status_dictionary)
print("\n",
    claim_status_dictionary.get("Approved", 0)
)


{'Medical', 'Life', 'Vehicle'}

UNIQUE CLAIM CATEGORIES
{'Accident', 'Other', 'Damage', 'Medical', 'Theft', 'Loss'}

CLAIM COUNTS BY STATUS
{'Approved': 506, 'Rejected': 123, 'Pending': 121}

 506


Question 12. Write a function that classifies claims as Small, Medium, or Large using stated claim-amount ranges.

In [156]:


def classify_claim(amount):
  if amount < 150000:
    return "Small"
  elif amount < 350000:
    return "Medium"
  else:
    return "Large"

df["Claim_Size"] = df["Claim_Amount"].apply(
    classify_claim
)

print("\nCLAIM SIZE CLASSIFICATION")
print(df[["Claim_Amount", "Claim_Size"]])



CLAIM SIZE CLASSIFICATION
     Claim_Amount Claim_Size
0       393052.81      Large
1       268590.57     Medium
2       320110.30     Medium
3       488247.47      Large
4       483651.10      Large
..            ...        ...
745     170375.34     Medium
746     231509.64     Medium
747     248206.64     Medium
748     277995.32     Medium
749      92818.41      Small

[750 rows x 2 columns]


Question 13. Use NumPy to calculate mean, median, and standard deviation of claim amounts.


In [157]:


claim_amounts = df["Claim_Amount"].dropna().to_numpy()
mean_amount = np.mean(claim_amounts)
median_amount = np.median(claim_amounts)
std_amount = np.std(claim_amounts)

print("\nCLAIM AMOUNT STATISTICS")
print("Mean:", round(mean_amount, 2))
print("Median:", round(median_amount, 2))
print("Standard Deviation:", round(std_amount, 2))


CLAIM AMOUNT STATISTICS
Mean: 248577.01
Median: 250817.91
Standard Deviation: 144295.95


Question 14. Identify unusually high claim amounts using the IQR method and compare average settlement days by claim status.

In [159]:


Q1 = np.percentile(claim_amounts, 25)
Q3 = np.percentile(claim_amounts, 75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

high_claim_outliers = df[
    df["Claim_Amount"] > upper_limit
]

print("\nHIGH-VALUE CLAIM OUTLIERS")
print(
    high_claim_outliers[
        ["Claim_ID", "Claim_Amount", "Insurance_Type"]
    ]
)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)


settlement_by_status = df.groupby(
    "Claim_Status_Clean"
)["Settlement_Days"].mean()

print("\nAVERAGE SETTLEMENT DAYS BY STATUS")
print(settlement_by_status.round(2))




HIGH-VALUE CLAIM OUTLIERS
Empty DataFrame
Columns: [Claim_ID, Claim_Amount, Insurance_Type]
Index: []
Q1: 122367.24
Q3: 371144.10250000004
IQR: 248776.86250000005
Lower Limit: -250798.05375000008
Upper Limit: 744309.3962500001

AVERAGE SETTLEMENT DAYS BY STATUS
Claim_Status_Clean
Approved    59.34
Pending     63.51
Rejected    64.09
Name: Settlement_Days, dtype: float64


Question 15. Create a simple InsuranceClaim class with claim ID, claim type, and amount attributes, and write five process-improvement recommendations.

In [160]:


class InsuranceClaim:
    def __init__(self, claim_id, claim_type, amount):
        self.claim_id = claim_id
        self.claim_type = claim_type
        self.amount = amount

claim1 = InsuranceClaim(
    "CL001",
    "Hospitalisation",
    50000
)

print("\nINSURANCE CLAIM OBJECT")
print("Claim ID:", claim1.claim_id)
print("Claim Type:", claim1.claim_type)
print("Claim Amount:", claim1.amount)






INSURANCE CLAIM OBJECT
Claim ID: CL001
Claim Type: Hospitalisation
Claim Amount: 50000


# Process-improvement recommendations

1. **Set a settlement-time target and prioritise the slowest claim types.**  
**Finding:** Average settlement time differs by claim type, from 52.0 days for Damage to 66.0 days for Loss. Overall, 51.2% of claims take longer than 60 days and 24.8% take longer than 90 days.  
**Action:** Set a 60-day target, add extra handlers or automation for shorter waits and a smaller share of slow claims.

2. **Clear the pending backlog with an ageing and escalation rule.**  
**Finding:** 121 claims (16.1%) are still Pending, and they average 63.5 days.  
**Action:** Flag any claim pending beyond a set number of days and assign it an owner for a smaller backlog and no claims left waiting indefinitely.

3. **Reduce avoidable rejections, and decide rejections faster.**  
**Finding**: 123 claims (16.4%) were rejected. Rejection rates are similar across insurance types (13.2% for Motor to 18.4% for Health).  
**Action:** Add a pre-submission document checklist for every claim, and record a rejection reason for each rejected claim for fewer avoidable rejections, quicker decisions and less rework.

4. **Review high-value claims by value, not by statistical outlier.**  
**Finding:** The IQR method flagged no unusually high claims (upper limit 744,309, largest claim 499,443). Travel has the highest total claimed amount (42.28 million) and the most claims (164).  
**Action:** Add a senior-review step for claims above a fixed value (for example the top 5%), and keep an extra eye on Travel for a closer control where the money is, without slowing ordinary claims.

5. **Improve data quality at the point of entry.**  
**Finding:** The raw file had 11 duplicate Claim_IDs, 18 missing claim amounts and 18 missing settlement days. Also, 121 Pending and 122 Rejected claims still show settlement days, which shouldn’t exist for unsettled claims.  
**Action:** Make Claim_ID unique and amount mandatory in the claims system, and record settlement days only when a claim is actually settled, it results in Reliable reporting.

In [161]:
# Saving Cleaned Dataset
cleaned_output_path = "27_insurance_claims_cleaned.csv"
df.to_csv(cleaned_output_path, index=False)
print(f"\nSaved cleaned dataset to {cleaned_output_path}")


Saved cleaned dataset to 27_insurance_claims_cleaned.csv
